# W8_00 : Bayesian Optimization
Pre-lab: Try three methods to find the best formulation for a simulated electrolyte. Compare grid search, random search, and Bayesian optimization (BO). Then write the code for two BO acquisition functions, UCB and EI.

Fill in the blanks: Replace each ___ with your code, then run the cell. If you need a hint, open the 💡 cell below.

In [ ]:
#@title 1. Setup — click ▶ to start
# Helper code for this lab. Nothing to edit here.

import warnings

import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import norm
from sklearn.exceptions import ConvergenceWarning
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import ConstantKernel, Matern, WhiteKernel

# ---------- blanks (___) and ✅/❌ messages ----------
from IPython import get_ipython
from IPython.display import HTML, display


class BlankError(Exception):
    """A ___ is still in the cell."""


class _Blank:
    __array_ufunc__ = None  # numpy hands arithmetic back to us -> friendly stop

    def __repr__(self):
        return '___'

    def _stop(self, *args, **kwargs):
        raise BlankError('Replace every ___ in this cell with your answer, then run the cell again.')

    __add__ = __radd__ = __sub__ = __rsub__ = __mul__ = __rmul__ = __truediv__ = __rtruediv__ = _stop
    __pow__ = __rpow__ = __neg__ = __float__ = __int__ = __index__ = __bool__ = _stop
    __lt__ = __le__ = __gt__ = __ge__ = __call__ = __getitem__ = __iter__ = __len__ = _stop


___ = _Blank()


def has_blank(v):
    if isinstance(v, _Blank):
        return True
    if isinstance(v, dict):
        return any(has_blank(x) for x in v.values())
    if isinstance(v, (list, tuple)):
        return any(has_blank(x) for x in v)
    return False


def need(**values):
    """Stop with a friendly message if any value is (or contains) ___."""
    for name, v in values.items():
        if has_blank(v):
            raise BlankError(f'Fill in <b>{name}</b> (it still has ___), then run the cell again.')


def _say(color, text):
    display(HTML(f'<div style="border-left:5px solid {color};padding:6px 12px;margin:6px 0;font-size:14px">{text}</div>'))


def ok(text):
    _say('#3A9D5D', '✅ ' + text)


def bad(text):
    _say('#D9534F', '❌ ' + text)


def _friendly(shell, etype, evalue, tb, tb_offset=None):
    _say('#E8833A', '✋ ' + str(evalue))


if get_ipython() is not None:
    get_ipython().set_custom_exc((BlankError,), _friendly)

C_LIM, PHI_LIM = (0.2, 2.4), (0.0, 1.0)  # salt molality (mol/kg), EC fraction in EC/DMC
SLICE_C = 1.9                            # 1-D slice used for the EI/UCB demo
START = (0.1, 0.3, 0.5)                  # three EC fractions already measured on the slice
ORANGE, BLUE, GREEN, GREY = '#E8833A', '#2F6DB5', '#3A9D5D', '#8C929B'
COLORS = {'Grid': GREY, 'Random': GREEN, 'BO': BLUE}
STRATEGIES = ('Grid', 'Random', 'BO')


def casteel_amis(c, s_max, mu, a=0.95, b=-0.05):
    return s_max * (c / mu) ** a * np.exp(b * (c - mu) ** 2 - a / mu * (c - mu))


def conductivity(c, phi):
    """Ionic conductivity (mS/cm) of the toy electrolyte."""
    s_max = 11.5 * (0.35 + 0.65 * np.exp(-((phi - 0.45) / 0.28) ** 2))
    mu = 0.8 + 0.6 * phi
    spot = 9.0 * np.exp(-((phi - 0.85) ** 2 / (2 * 0.12 ** 2) + (c - 1.9) ** 2 / (2 * 0.35 ** 2)))
    return casteel_amis(c, s_max, mu) + spot


def unit(X):
    return (X - [C_LIM[0], PHI_LIM[0]]) / [C_LIM[1] - C_LIM[0], PHI_LIM[1] - PHI_LIM[0]]


def fit_gp(X, y):
    kernel = (ConstantKernel(1.0, (1e-2, 1e2)) * Matern(length_scale=0.2, length_scale_bounds=(0.05, 1.0), nu=2.5)
              + WhiteKernel(1e-6, (1e-8, 1e-2)))
    gp = GaussianProcessRegressor(kernel, normalize_y=True, n_restarts_optimizer=1, random_state=0)
    with warnings.catch_warnings():
        warnings.simplefilter('ignore', ConvergenceWarning)
        gp.fit(X, y)
    return gp


def ucb_ref(mu, sd, beta):
    return mu + np.sqrt(beta) * sd


def ei_ref(mu, sd, best, xi):
    z = (mu - best - xi) / sd
    return (mu - best - xi) * norm.cdf(z) + sd * norm.pdf(z)


def acquisition(mu, sd, best, acq='EI', beta=4.0, xi=0.01, ucb=None, ei=None):
    sd = np.maximum(sd, 1e-12)
    if acq == 'UCB':
        return (ucb or ucb_ref)(mu, sd, beta)
    return (ei or ei_ref)(mu, sd, best, xi)


# ---------- the landscape ----------
GRID_C = np.linspace(*C_LIM, 120)
GRID_PHI = np.linspace(*PHI_LIM, 100)
CC, PP = np.meshgrid(GRID_C, GRID_PHI)
TRUTH = conductivity(CC, PP)
DENSE = np.column_stack([CC.ravel(), PP.ravel()])
PHI = np.linspace(*PHI_LIM, 400)


def heat(ax, Z, title, cmap='viridis', vmin=None, vmax=None):
    im = ax.imshow(Z, origin='lower', extent=[*C_LIM, *PHI_LIM], aspect='auto', cmap=cmap, vmin=vmin, vmax=vmax)
    ax.set(xlabel='salt (mol/kg)', ylabel='EC fraction', title=title)
    return im


def show_landscape():
    fig, ax = plt.subplots(1, 3, figsize=(16, 4.4), gridspec_kw={'width_ratios': [1, 1.15, 1]})
    c = np.linspace(*C_LIM, 300)
    for phi, col in [(0.1, GREY), (0.45, BLUE), (0.85, ORANGE)]:
        ax[0].plot(c, conductivity(c, phi), color=col, lw=2, label=f'EC fraction {phi}')
    ax[0].set(xlabel='salt (mol/kg)', ylabel='conductivity (mS/cm)', title='(a) Conductivity vs salt: rise, peak, fall')
    ax[0].legend(fontsize=8)

    im = heat(ax[1], TRUTH, '(b) True landscape (unknown to the optimizer)')
    fig.colorbar(im, ax=ax[1], label='mS/cm')
    i = np.unravel_index(TRUTH.argmax(), TRUTH.shape)
    ax[1].plot(CC[i], PP[i], '*', color='white', ms=14, mec='black')
    ax[1].annotate('small global best', (CC[i], PP[i]), (1.05, 0.93), color='white', fontsize=9,
                   arrowprops=dict(arrowstyle='->', color='white'))
    ax[1].annotate('broad local best', (1.07, 0.45), (0.3, 0.12), color='white', fontsize=9,
                   arrowprops=dict(arrowstyle='->', color='white'))
    ax[1].axvline(SLICE_C, color='white', ls='--', lw=1)

    ax[2].plot(PHI, conductivity(SLICE_C, PHI), color='black', lw=2)
    ax[2].set(xlabel='EC fraction', ylabel='conductivity (mS/cm)', title=f'(c) Slice at {SLICE_C} mol/kg (used below)')
    fig.tight_layout()
    plt.show()
    print(f'Global best: {TRUTH.max():.2f} mS/cm at salt {CC[i]:.2f} mol/kg, EC fraction {PP[i]:.2f}')


# ---------- you are the optimizer ----------
def bo_slice(acq, steps, start=START, beta=4.0, xi=0.01):
    X = np.array(start, float)
    for _ in range(steps):
        y = conductivity(SLICE_C, X)
        mu, sd = fit_gp(X[:, None], y).predict(PHI[:, None], return_std=True)
        X = np.append(X, PHI[np.argmax(acquisition(mu, sd, y.max(), acq, beta, xi))])
    return X


class HumanRun:
    """You choose the next EC fraction on the 1.9 mol/kg slice; 5 experiments."""

    def __init__(self, start=START, budget=5):
        self.start, self.budget = list(start), budget
        self.restart()

    def restart(self):
        self.x = list(self.start)

    @property
    def used(self):
        return len(self.x) - len(self.start)

    def measure(self, x):
        need(NEXT=x)
        x = float(x)
        if not 0 <= x <= 1:
            return bad('Choose an EC fraction between 0 and 1.')
        if self.used >= self.budget:
            return bad(f'All {self.budget} experiments are used. Run the next cell to see how you did '
                       '(or <code>me.restart()</code> to play again).')
        self.x.append(x)
        y = conductivity(SLICE_C, np.array(self.x))
        fig, ax = plt.subplots(figsize=(7, 3.6))
        n0 = len(self.start)
        ax.scatter(self.x[:n0], y[:n0], color=GREY, s=50, label='already measured')
        ax.scatter(self.x[n0:], y[n0:], color=ORANGE, s=70, edgecolors='black', label='your experiments', zorder=3)
        for k, (a, b) in enumerate(zip(self.x[n0:], y[n0:]), 1):
            ax.annotate(str(k), (a, b), (5, 6), textcoords='offset points', fontsize=10, weight='bold')
        ax.set(xlim=(-0.02, 1.02), ylim=(2, 15), xlabel='EC fraction', ylabel='conductivity (mS/cm)',
               title=f'Experiment {self.used}/{self.budget}: EC {x:.2f} → {y[-1]:.2f} mS/cm   (best so far {y.max():.2f})')
        ax.legend(loc='upper left', fontsize=8)
        plt.show()
        if self.used == self.budget:
            ok('All experiments used. Run the next cell to compare with BO.')

    def reveal(self):
        if self.used < self.budget:
            return bad(f'You have used {self.used}/{self.budget} experiments. Go back and finish them first.')
        runs = {'You': np.array(self.x), 'BO with EI': bo_slice('EI', self.budget), 'BO with UCB (β=4)': bo_slice('UCB', self.budget)}
        true = conductivity(SLICE_C, PHI)
        fig, ax = plt.subplots(1, 3, figsize=(15, 3.8), sharey=True)
        n0 = len(self.start)
        for a, (name, X) in zip(ax, runs.items()):
            y = conductivity(SLICE_C, X)
            a.plot(PHI, true, color='black', lw=1, ls=':', label='truth (now revealed)')
            a.scatter(X[:n0], y[:n0], color=GREY, s=40)
            a.scatter(X[n0:], y[n0:], color=ORANGE if name == 'You' else BLUE, s=60, edgecolors='black', zorder=3)
            for k, (p, q) in enumerate(zip(X[n0:], y[n0:]), 1):
                a.annotate(str(k), (p, q), (5, 6), textcoords='offset points', fontsize=9)
            a.set(title=f'{name}: best {y.max():.2f} mS/cm', xlabel='EC fraction', ylim=(2, 15))
        ax[0].set_ylabel('conductivity (mS/cm)')
        ax[0].legend(loc='upper left', fontsize=8)
        fig.tight_layout()
        plt.show()
        mine = conductivity(SLICE_C, runs['You']).max()
        bo = max(conductivity(SLICE_C, X).max() for k, X in runs.items() if k != 'You')
        if mine >= bo - 0.05:
            ok(f'You matched BO ({mine:.2f} vs {bo:.2f} mS/cm). Could you explain your strategy as a rule a computer could follow?')
        else:
            bad(f'BO beat you this time ({bo:.2f} vs {mine:.2f} mS/cm). Did you spend experiments near points you already knew?')


# ---------- three strategies ----------
def grid_points(n):
    nc = int(np.floor(np.sqrt(n * 1.2)))
    nphi = n // nc
    c, p = np.meshgrid(np.linspace(*C_LIM, nc), np.linspace(*PHI_LIM, nphi))
    return np.column_stack([c.ravel(), p.ravel()])


def random_points(n, seed):
    rng = np.random.default_rng(seed)
    return np.column_stack([rng.uniform(*C_LIM, n), rng.uniform(*PHI_LIM, n)])


def bo_points(n, seed, acq='EI', n_init=5, beta=4.0, xi=0.01):
    X = random_points(n_init, seed)
    y = conductivity(*X.T)
    while len(X) < n:
        gp = fit_gp(unit(X), y)
        mu, sd = gp.predict(unit(DENSE), return_std=True)
        x = DENSE[np.argmax(acquisition(mu, sd, y.max(), acq, beta, xi))]
        X, y = np.vstack([X, x]), np.append(y, conductivity(*x))
    return X


def strategy_points(name, n, seed, acq='EI'):
    return {'Grid': lambda: grid_points(n), 'Random': lambda: random_points(n, seed),
            'BO': lambda: bo_points(n, seed, acq)}[name]()


def compare_strategies(budget=20, seed=0, acq='EI', winner=None, best_r2=None):
    """Same number of experiments for each strategy; then fit a GP surrogate to each set."""
    need(MY_WINNER=winner, MY_BEST_R2=best_r2, BUDGET=budget)
    for name, v in [('MY_WINNER', winner), ('MY_BEST_R2', best_r2)]:
        if v is not None and v not in STRATEGIES:
            return bad(f"{name} must be 'Grid', 'Random' or 'BO' (with quotes).")
    fig, ax = plt.subplots(3, 3, figsize=(15, 12))
    rows = {}
    for k, name in enumerate(STRATEGIES):
        X = strategy_points(name, int(budget), seed, acq)
        y = conductivity(*X.T)
        mean = fit_gp(unit(X), y).predict(unit(DENSE)).reshape(TRUTH.shape)
        err = np.abs(mean - TRUTH)
        r2 = 1 - ((mean - TRUTH) ** 2).sum() / ((TRUTH - TRUTH.mean()) ** 2).sum()
        heat(ax[k, 0], TRUTH, f'{name}: {len(X)} experiments' + (' (□ = 5 random starts)' if name == 'BO' else ''))
        if name == 'BO':
            ax[k, 0].scatter(*X[:5].T, marker='s', c='white', edgecolors='black', s=36, clip_on=False)
        ax[k, 0].scatter(*(X[5:] if name == 'BO' else X).T, c='white', edgecolors='black', s=36, clip_on=False)
        ax[k, 0].plot(*X[np.argmax(y)], '*', color=ORANGE, ms=16, mec='black')
        heat(ax[k, 1], mean, f'{name}: GP surrogate (R² = {r2:.2f})', vmin=TRUTH.min(), vmax=TRUTH.max())
        heat(ax[k, 2], err, f'{name}: |surrogate − truth|', cmap='magma', vmin=0, vmax=6)
        rows[name] = (y.max(), r2, err.mean())
    fig.tight_layout()
    plt.show()
    print(f'{"strategy":8s} {"best found":>11s} {"surrogate R²":>13s} {"mean |error|":>13s}   (true best {TRUTH.max():.2f})')
    for name, (best, r2, e) in rows.items():
        print(f'{name:8s} {best:11.2f} {r2:13.2f} {e:13.2f}')
    top = max(rows, key=lambda k: rows[k][0])
    top_r2 = max(rows, key=lambda k: rows[k][1])
    if winner is not None:
        (ok if winner == top else bad)(f'Highest conductivity: <b>{top}</b> (you said {winner}). '
                                       'BO spends its experiments where the surrogate looks promising or is unsure.')
    if best_r2 is not None:
        (ok if best_r2 == top_r2 else bad)(f'Surrogate most like the truth: <b>{top_r2}</b> (you said {best_r2}). '
                                           'R² scores the whole map; BO only needs the map to be right near the best region.')


def convergence(budget=30, seeds=10, acq='EI'):
    """Best value found vs number of experiments, over several random starts."""
    n = np.arange(1, budget + 1)
    fig, ax = plt.subplots(figsize=(7, 4.4))
    for name in ['Random', 'BO']:
        curves = np.array([np.maximum.accumulate(conductivity(*strategy_points(name, budget, s, acq).T))
                           for s in range(seeds)])
        ax.plot(n, np.median(curves, 0), color=COLORS[name], lw=2, label=f'{name} (median of {seeds} runs)')
        ax.fill_between(n, *np.percentile(curves, [25, 75], 0), color=COLORS[name], alpha=0.2)
    sizes = [4, 9, 16, 20, 25, 30]
    ax.plot(sizes, [conductivity(*grid_points(m).T).max() for m in sizes], 's--', color=COLORS['Grid'], label='Grid (whole grid)')
    ax.axhline(TRUTH.max(), color='black', ls=':', label='true best')
    ax.set(xlabel='number of experiments', ylabel='best conductivity found (mS/cm)', title='How fast each strategy finds a good electrolyte')
    ax.legend(fontsize=8, loc='lower right')
    plt.show()


# ---------- acquisition functions ----------
def _try(f, *args):
    try:
        out = f(*args)
    except BlankError:
        raise
    except Exception as e:  # show students the problem without a long traceback
        bad(f'Your function raised {type(e).__name__}: {e}')
        return None
    need(**{'the blank in your function': out})
    return np.asarray(out, float)


def check_ucb(f):
    mu, sd = np.array([0.0, 1.0, 2.0]), np.array([1.0, 0.5, 0.0])
    out = _try(f, mu, sd, 4.0)
    if out is None:
        return
    want = ucb_ref(mu, sd, 4.0)
    if out.shape == want.shape and np.allclose(out, want):
        ok('UCB is correct. With β = 4 all three points tie: a sure 2.0 is worth the same as an uncertain 0 ± 1.')
    else:
        bad(f'For mu={mu.tolist()}, sd={sd.tolist()}, β=4 we expect {want.tolist()} but got {np.round(out, 3).tolist()}. '
            'Hint: mean + np.sqrt(beta) * std.')


def check_ei(f):
    mu, sd = np.array([0.5, 1.0, 1.5]), np.array([1.0, 0.2, 0.5])
    out = _try(f, mu, sd, 1.0, 0.01)
    if out is None:
        return
    want = ei_ref(mu, sd, 1.0, 0.01)
    if out.shape == want.shape and np.allclose(out, want):
        ok('EI is correct. The first term rewards a high mean; the second term rewards uncertainty.')
    else:
        bad(f'Expected {np.round(want, 4).tolist()} but got {np.round(out, 4).tolist()}. '
            'Hint: the second term is (uncertainty) × norm.pdf(z).')


def step_by_step(steps=5, start=START, beta=4.0, xi=0.01, ucb=None, ei=None):
    """1-D slice, EI next to UCB: GP mean ± 2σ (top) and acquisition (bottom); ★ = next experiment."""
    need(beta=beta, xi=xi)
    _steps(steps, start, beta, xi, ucb, ei, ('EI', 'UCB'))


def _steps(steps, start, beta, xi, ucb, ei, acqs):
    true = conductivity(SLICE_C, PHI)
    fig = plt.figure(figsize=(6.5 * len(acqs), 3.1 * steps))
    outer = fig.add_gridspec(steps, len(acqs), hspace=0.45, wspace=0.18)
    best = {}
    for col, acq in enumerate(acqs):
        X = np.array(start, float)
        for k in range(steps):
            y = conductivity(SLICE_C, X)
            mu, sd = fit_gp(X[:, None], y).predict(PHI[:, None], return_std=True)
            a = np.asarray(acquisition(mu, sd, y.max(), acq, beta, xi, ucb, ei), float)
            nxt = PHI[np.argmax(a)]
            inner = outer[k, col].subgridspec(2, 1, height_ratios=[2.2, 1], hspace=0.05)
            top, bot = fig.add_subplot(inner[0]), fig.add_subplot(inner[1])
            top.plot(PHI, true, ':', color='black', lw=1.2, label='truth (hidden)')
            top.plot(PHI, mu, color=BLUE, lw=1.8, label='GP mean')
            top.fill_between(PHI, mu - 2 * sd, mu + 2 * sd, color=BLUE, alpha=0.15, label='±2σ')
            n_old = len(start) if k == 0 else len(X) - 1
            top.scatter(X[:n_old], y[:n_old], color='black', s=22, zorder=3, label='measured')
            top.scatter(X[n_old:], y[n_old:], color=ORANGE, s=40, zorder=4, edgecolors='black', label='newest')
            top.axvline(nxt, color=ORANGE, lw=1, ls='--')
            top.set_xticks([])
            top.set_ylim(2, 17)
            name = f'EI (ξ={xi})' if acq == 'EI' else f'UCB (β={beta})'
            top.set_title(f'{name} · step {k + 1}: {len(X)} points, best {y.max():.2f}', fontsize=10)
            bot.fill_between(PHI, 0, a - a.min(), color=GREEN, alpha=0.35)
            bot.plot(nxt, (a - a.min()).max(), '*', color=ORANGE, ms=13, mec='black')
            bot.set_yticks([])
            bot.set_ylabel(acq, fontsize=9)
            if k == steps - 1:
                bot.set_xlabel('EC fraction')
            else:
                bot.set_xticks([])
            if k == 0 and col == 0:
                top.legend(fontsize=7, loc='upper left', ncol=2)
            X = np.append(X, nxt)
        best[acq] = conductivity(SLICE_C, X).max()
    plt.show()
    return best


def beta_challenge(beta, steps=4, target=13.0):
    """Can UCB reach the sweet spot within 4 steps from START?"""
    need(BETA=beta)
    if float(beta) <= 0:
        return bad('β must be a positive number.')
    best = _steps(steps, START, float(beta), 0.01, None, None, ('UCB',))['UCB']
    if best >= target:
        ok(f'β = {beta}: UCB reached {best:.2f} mS/cm in {steps} steps. Enough exploration to leave the broad peak.')
    else:
        bad(f'β = {beta}: best {best:.2f} mS/cm — UCB stayed near the broad peak. Try a larger β (more exploration).')

plt.rcParams['figure.dpi'] = 100
me = HumanRun()
print('Ready.')

## 2. Toy example: electrolyte conductivity
This simulated map shows how electrolyte conductivity changes with salt concentration and EC fraction. It includes a small region where conductivity is higher than expected. Since we know where the best point is, we can check how close each method gets.

In [ ]:
show_landscape()

## 3. Find the best EC fraction by hand
The salt concentration is fixed at 1.9 mol/kg. The three grey points show the measurements you start with. Run the cell 5 times. Before each run, choose a new NEXT value (EC fraction from 0 to 1). Use the results so far to decide where to try next for higher conductivity.

To start over, run me.restart().

In [ ]:
NEXT = ___      # EC fraction for your next experiment, between 0 and 1 (e.g. 0.75)
me.measure(NEXT)

The hidden curve, and what BO would have picked:

In [ ]:
me.reveal()

## 4. Which search method will find the highest conductivity?
Grid search, random search, and Bayesian optimization (BO) each test 20 combinations of salt concentration and EC fraction. Before running the cell, enter your prediction using 'Grid', 'Random', or 'BO'.

Then read the three plots from left to right.
1. Tested points — The ★ marks the best result that method found.
2. Predicted conductivity — A Gaussian process (GP) estimates conductivity across the map using those 20 results.
3. Prediction error — See where the GP’s predictions differ from the true conductivity.

In [ ]:
MY_WINNER = ___     # who finds the highest conductivity?
MY_BEST_R2 = ___    # whose surrogate looks most like the true map (highest R²)?
compare_strategies(budget=20, winner=MY_WINNER, best_r2=MY_BEST_R2)

Same, over 10 random starts (~20 s):

In [ ]:
convergence(budget=30, seeds=10)

## 5. Write the functions that choose the next experiment
An acquisition function gives each candidate a score. BO tests the candidate with the highest score.

### 5-1. Upper confidence bound (UCB)

UCB gives a higher score to candidates with higher predicted conductivity or greater uncertainty.

Fill in ___ to calculate the UCB score for each candidate.

In [ ]:
def ucb(mu, sd, beta):
    return ___          # mean + sqrt(beta) * std   (use np.sqrt)

check_ucb(ucb)

In [ ]:
#@title 💡 Answer — double-click to show (run it if you are stuck)
def ucb(mu, sd, beta):
    return mu + np.sqrt(beta) * sd

check_ucb(ucb)

### **5-2. EI**

EI scores each candidate by how much it is expected to improve on the best conductivity measured so far (best).

Fill in ___ to calculate the EI score for each candidate.

In [ ]:
def ei(mu, sd, best, xi):
    z = (mu - best - xi) / sd
    return (mu - best - xi) * norm.cdf(z) + ___ * norm.pdf(z)   # what multiplies norm.pdf(z)? (the uncertainty)

check_ei(ei)

In [ ]:
#@title 💡 Answer — double-click to show (run it if you are stuck)
def ei(mu, sd, best, xi):
    z = (mu - best - xi) / sd
    return (mu - best - xi) * norm.cdf(z) + sd * norm.pdf(z)

check_ei(ei)

### **5-3. See how UCB and EI choose the next experiment**

The top plot shows the GP mean with a ±2σ uncertainty band. The bottom plot shows the acquisition scores, and ★ marks the next experiment.

In [ ]:
step_by_step(steps=5, beta=4.0, xi=0.01, ucb=ucb, ei=ei)   # uses YOUR ucb and ei

## 6. Can you reach the target in 4 steps?
Try different values of β in UCB. Find one that reaches a conductivity of 13 mS/cm or higher within 4 steps.

In [ ]:
BETA = ___      # try a number, e.g. 0.1, 1, 4, 25
beta_challenge(BETA)

In [ ]:
#@title 💡 Answer — double-click to show (run it if you are stuck)
BETA = 4.0   # any beta from about 1 upward works here; 0.1 and 0.5 stay on the broad peak
beta_challenge(BETA)